In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pylab as plt

import seaborn as sns

from skspatial.objects import Line, Plane
from skspatial.plotting import plot_3d

from skspatial.objects import Line, Cylinder, Point, Points
from skspatial.plotting import plot_3d

import phasespace

import tensorflow

import os

import bisect
import numpy as np
import matplotlib.pylab as plt
import pandas as pd

import seaborn as sns

import numpy as np
from sklearn.mixture import GaussianMixture
from scipy.stats import multivariate_normal

import numpy as np
from scipy.interpolate import griddata
from scipy.integrate import quad, trapezoid
from scipy.interpolate import CubicSpline

import matplotlib.pylab as plt
from scipy import stats
from matplotlib import cm
from matplotlib.ticker import LinearLocator

from scipy.interpolate import LinearNDInterpolator

import eloss_tools


import dm_generation_tools as dgt
import detector_simulation_tools as dst
import diagnostics as dg

import earthshine_io as eio
import diagnostics_v2 as diag


import glob

import time

####################################
import warnings
# Suppress all warnings
warnings.filterwarnings("ignore")


import pickle

In [ ]:
#infile = 'OUTPUT_FILES/generated_data_depth_-8.0--4000.0_diskR_4000.0_mDM_1000.0-9000.0_mA_0.22_dmModel_floating_HIT_DETECTOR_ave_eloss__COMBINED.parquet'
infile = 'OUTPUT_FILES/generated_data_depth_-8.0--4000.0_diskR_4000.0_mDM_10000.0-90000.0_mA_0.22_dmModel_floating_HIT_DETECTOR_ave_eloss__COMBINED.parquet'

df = pd.read_parquet(infile)

df

In [ ]:
df.info()

In [ ]:
mass = 1000
filter = (df['M_DM'] == mass)

len(df[filter])

In [ ]:
def get_df_of_acceptances(df, energycut=[100]):
    masses = df['M_DM'].unique()
    masses
    
    org_nevents = df['total_org_nevents'].iloc[0]
    print(org_nevents, org_nevents/1e6)

    dftot = None
    for idx,ecut in enumerate(energycut):
        # Require the muons to reach CMS with some given energy
        filter = (df['efinal_mu1']>ecut)

        # Require them to hit the inner detector
        filter_hit_id = df['hit_inner_detector']==True

        #########################################################################
        vcounts = df[filter]['M_DM'].value_counts()
        dftmp = vcounts.to_frame()
        dftmp = dftmp.reset_index(names='M_DM')
        dftmp = dftmp.rename(columns={'count':f'count_ecut{ecut}'})
        
        dftmp['org_nevents'] = org_nevents
        dftmp[f'frac_ecut{ecut}'] = dftmp[f'count_ecut{ecut}']/org_nevents

        #########################################################################
        vcounts = df[filter & filter_hit_id]['M_DM'].value_counts()
        dftmp2 = vcounts.to_frame()
        dftmp2 = dftmp2.reset_index(names='M_DM')
        dftmp2 = dftmp2.rename(columns={'count':f'count_hit_id_ecut{ecut}'})
        
        dftmp2['org_nevents'] = org_nevents
        dftmp2[f'frac_hit_id_ecut{ecut}'] = dftmp2[f'count_hit_id_ecut{ecut}']/org_nevents

        
        print(ecut)

        if idx==0:
            dftot = dftmp.copy()
            dftot = pd.merge(dftot, dftmp2, on=['M_DM', 'org_nevents'], how='outer')

        else:
            #dftot = pd.concat([dftmp, dftot], join='outer', ignore_index=True)
            dftot = pd.merge(dftot, dftmp, on=['M_DM', 'org_nevents'], how='outer')
            dftot = pd.merge(dftot, dftmp2, on=['M_DM', 'org_nevents'], how='outer')
            
    
    #dftmp.columns
    #dftmp.index
    
    #dftmp.values
    #dftmp
    return dftot

In [ ]:
#generated_data_depth_-8.0--4000.0_diskR_4000.0_mDM_1000.0-9000.0_mA_0.22_dmModel_floating_HIT_DETECTOR_ave_eloss__COMBINED.parquet

#model = 'dmModel_momentum'
model = 'dmModel_floating'
vol = 'depth_-8.0--4000.0_diskR_4000.0'
volume = (4000-8)*(np.pi*(4000**2))

'''
vol = 'depth_-8.0--4000.0_diskR_40.0'
volume = (4000-8)*(np.pi*(40**2))
model = 'dmModel_core'
'''

print(f'volume: {volume} m^3')
print(f'volume: {volume/1e9} km^3')


eloss = 'ave_eloss'

my_dir = './OUTPUT_FILES/'
infiles = glob.glob(my_dir + f"/*{vol}*{model}*{eloss}*COMBINED.parquet")

#infiles = glob

dfs = []
for infile in infiles:
    
    print(infile)
    df = pd.read_parquet(infile)

    dftmp = get_df_of_acceptances(df, energycut=[10, 100, 1000])

    dfs.append(dftmp)

dfs

dfacc = pd.concat(dfs)
dfacc = dfacc.sort_values(by='M_DM')

dfacc['volume m3'] = volume 

dfacc

In [ ]:
#model = 'dmModel_floating'
#model = 'dmModel_momentum'
#model = 'dmModel_floating'
#model = 'dmModel_core'

label_tag = ''
ylim = (0,1)
if model == 'dmModel_core':
    label_tag = 'core'
    ylim = (1e-4,0.2)
elif model == 'dmModel_floating':
    label_tag = 'floating'
    ylim = (2e-9,2e-5)
elif model == 'dmModel_momentum':
    label_tag = 'mono-energetic'
    ylim = (2e-9,2e-5)

plt.figure(figsize=(8,4))
dfacc.plot(x='M_DM', y='frac_ecut10', kind='scatter', ax=plt.gca(), color='b', marker='o', s=30,  label=r'$E_{\mu}$ > 10 GeV')
dfacc.plot(x='M_DM', y='frac_ecut100', kind='scatter', ax=plt.gca(), color='r', marker='^',s=30, label='$E_{\mu}$ > 100 GeV')
dfacc.plot(x='M_DM', y='frac_ecut1000', kind='scatter', ax=plt.gca(), color='g', marker='v',s=30, label='$E_{\mu}$ > 1000 GeV')

dfacc.plot(x='M_DM', y='frac_hit_id_ecut10', kind='scatter', ax=plt.gca(), color='b', marker='s', s=30,  label=r'$E_{\mu}$ > 10 GeV (ID)')
dfacc.plot(x='M_DM', y='frac_hit_id_ecut100', kind='scatter', ax=plt.gca(), color='r', marker='P',s=30, label='$E_{\mu}$ > 100 GeV (ID)')
dfacc.plot(x='M_DM', y='frac_hit_id_ecut1000', kind='scatter', ax=plt.gca(), color='g', marker='>',s=30, label='$E_{\mu}$ > 1000 GeV (ID)')


plt.title(f'DM model: {label_tag}     volume: {volume/1e9:.3f} km$^3$')
plt.xscale('log')
plt.xlabel('$M_{DM}$ GeV/c$^2$', fontsize=16)
#plt.ylabel('# $\mu$ strike detector / # $mu$', fontsize=16)
plt.ylabel('acceptance (frac)', fontsize=16)
plt.ylim(ylim[0], ylim[1])

plt.xlim(5e2, 1e9)

plt.legend(fontsize=12)
plt.yscale('log')
plt.tight_layout()

filename = f'acc_dm_model_{label_tag}.png'
plt.savefig(filename)

filename = f'acc_dm_model_{label_tag}.parquet'
dfacc.to_parquet(filename)

# Checking efficiences for different parameter settings

In [ ]:
cat = eio.read_catalog("data")

depth = -108.0

df, params = eio.load_many(cat, dm_model="momentum_constrained", stage="combined",
                      depth_min=depth, depth_max=depth, \
                     mDM_min=10000, mDM_max=10000, eloss='ave')   # add filters until exactly 1 matches

norg = df['total_org_nevents'].iloc[0]
filter = df['hit_inner_detector']==True
n = len(df[filter])

print(f'depth: {depth}   # org: {norg}    n: {n}      n/# org: {n/norg:.2e}      100*n/# org (%): {100*n/norg:.2e}  ')

In [ ]:
filter = (df['efinal_mu1']>10)


sns.histplot(df[filter], x='distance_to_detector', bins=100, hue='M_DM')


In [ ]:
filter = (df['efinal_mu1']>10)

sns.histplot(df[filter], x='y0', bins=100, hue='M_DM')


In [ ]:
filter = (df['efinal_mu1']>100)

sns.histplot(df[filter], x='y0', bins=100, hue='M_DM', binrange=(-4000,0))


In [ ]:
filter = (df['M_DM']==10000) & (df['efinal_mu1']>10)

df[filter]['y0'].hist(bins=100)